In [ ]:
!pip install -qU langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma pypdf chromadb gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/1

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

In [ ]:
current_retriever = None

def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        source = os.path.basename(source)
        page = doc.metadata.get("page", "Unknown")
        page_label = f"Page {int(page) + 1}" if str(page).isdigit() else f"Page {page}"
        formatted.append(f"[Source: {source} | {page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)

def get_unique_sources(docs):
    seen = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "Unknown"))
        page = doc.metadata.get("page", "Unknown")
        page_label = f"Page {int(page) + 1}" if str(page).isdigit() else f"Page {page}"
        label = f"{source} ({page_label})"
        if label not in seen:
            seen.append(label)
    return seen

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import uuid

def process_pdfs(files, chunk_size, chunk_overlap):
    global current_retriever, google_api_key

    if not google_api_key:
        return "System Message: API Key not found."
    if not files:
        return "System Message: Please upload at least one PDF file."

    documents = []
    for file in files:
        file_path = file if isinstance(file, str) else file.name if hasattr(file, 'name') else file.get('name') if isinstance(file, dict) else str(file)
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())

    if not documents:
        return "System Message: No text could be extracted."

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(chunk_size),
        chunk_overlap=int(chunk_overlap)
    )
    chunks = text_splitter.split_documents(documents)

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001",
        google_api_key=google_api_key
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=f"rag_{uuid.uuid4().hex[:8]}"
    )

    current_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    return f"System Message: Processing complete. Loaded {len(files)} files, {len(chunks)} chunks."

/tmp/ipykernel_1667/2782340921.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def ask_question(question, history):
    global current_retriever, google_api_key

    if not question.strip():
        return history, ""

    if current_retriever is None:
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": "Please process your PDF documents first."})
        return history, ""

    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-pro-latest",
        temperature=0,
        google_api_key=google_api_key
    )

    prompt = ChatPromptTemplate.from_template("""
You are a professional assistant analyzing document text.

Answer the user's question using ONLY the provided context below.
If the answer cannot be found in the context, output exactly: "I don't know." Do not invent information.

CRITICAL INSTRUCTIONS FOR TONE AND LANGUAGE:
1. You MUST answer the user in the EXACT SAME LANGUAGE they used to ask the question.
2. You MUST mirror the user's EXACT TONE, MANNER, and STYLE. For example, if the user speaks formally, respond formally. If they use slang, casual, or colloquial language, you must respond in the same casual slang or dialect. Adapt your persona completely to match how the user speaks to you.

Context:
{context}

Question:
{question}
""")

    docs = current_retriever.invoke(question)
    context = format_docs_with_sources(docs)

    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    answer = response.content

    sources = get_unique_sources(docs)
    if answer.strip().lower() not in ["i don't know.", "i don't know", '"i don\'t know."'] and sources:
        sources_text = "\n\n**Sources:**\n- " + "\n- ".join(sources)
        answer += sources_text

    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})

    return history, ""

def clear_all():
    global current_retriever
    current_retriever = None
    return [], "", None, "System Message: Ready."

In [ ]:
import gradio as gr

# ──────────────────────────────────────────────────────────────────────────────
# 🎨  Custom CSS — Premium Dark Theme with Glassmorphism
# ──────────────────────────────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

/* ── Global ─────────────────────────────────────────────────────────────── */
.gradio-container {
    font-family: 'Inter', sans-serif !important;
    background: linear-gradient(135deg, #0f0c29 0%, #1a1a3e 40%, #24243e 100%) !important;
    min-height: 100vh;
}

/* ── Header ─────────────────────────────────────────────────────────────── */
#app-header {
    text-align: center;
    padding: 28px 20px 18px;
    margin-bottom: 8px;
}
#app-header h1 {
    font-size: 2.2rem !important;
    font-weight: 700 !important;
    background: linear-gradient(135deg, #a78bfa, #818cf8, #60a5fa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    letter-spacing: -0.5px;
    margin: 0 !important;
}
#app-header p {
    color: #94a3b8 !important;
    font-size: 0.95rem !important;
    margin-top: 6px !important;
    font-weight: 400;
}

/* ── Glass Panels ───────────────────────────────────────────────────────── */
.panel-card {
    background: rgba(30, 32, 58, 0.65) !important;
    backdrop-filter: blur(18px) !important;
    -webkit-backdrop-filter: blur(18px) !important;
    border: 1px solid rgba(139, 135, 230, 0.15) !important;
    border-radius: 18px !important;
    padding: 24px !important;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.25);
    transition: border-color 0.3s ease, box-shadow 0.3s ease;
}
.panel-card:hover {
    border-color: rgba(139, 135, 230, 0.35) !important;
    box-shadow: 0 8px 40px rgba(99, 102, 241, 0.12);
}

/* ── Section Titles ─────────────────────────────────────────────────────── */
.section-title {
    font-size: 0.82rem !important;
    font-weight: 600 !important;
    text-transform: uppercase !important;
    letter-spacing: 1.4px !important;
    color: #a78bfa !important;
    margin-bottom: 14px !important;
    padding-bottom: 10px !important;
    border-bottom: 1px solid rgba(139, 135, 230, 0.15) !important;
}

/* ── Labels ──────────────────────────────────────────────────────────────── */
label, .label-wrap span {
    color: #c4b5fd !important;
    font-weight: 500 !important;
    font-size: 0.85rem !important;
}

/* ── File Upload ─────────────────────────────────────────────────────────── */
.upload-zone {
    border: 2px dashed rgba(139, 135, 230, 0.3) !important;
    border-radius: 14px !important;
    background: rgba(99, 102, 241, 0.04) !important;
    transition: all 0.3s ease !important;
    min-height: 100px !important;
}
.upload-zone:hover {
    border-color: rgba(139, 135, 230, 0.6) !important;
    background: rgba(99, 102, 241, 0.08) !important;
}

/* ── Number Inputs & Textboxes ───────────────────────────────────────────── */
input[type="number"],
textarea,
.textbox textarea,
.textbox input {
    background: rgba(15, 15, 35, 0.6) !important;
    border: 1px solid rgba(139, 135, 230, 0.2) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 0.9rem !important;
    padding: 10px 14px !important;
    transition: border-color 0.25s ease, box-shadow 0.25s ease !important;
}
input[type="number"]:focus,
textarea:focus,
.textbox textarea:focus,
.textbox input:focus {
    border-color: #818cf8 !important;
    box-shadow: 0 0 0 3px rgba(129, 140, 248, 0.15) !important;
    outline: none !important;
}

/* ── Primary Button ──────────────────────────────────────────────────────── */
button.primary {
    background: linear-gradient(135deg, #7c3aed, #6366f1, #818cf8) !important;
    border: none !important;
    border-radius: 12px !important;
    color: #fff !important;
    font-family: 'Inter', sans-serif !important;
    font-weight: 600 !important;
    font-size: 0.9rem !important;
    padding: 12px 24px !important;
    letter-spacing: 0.3px;
    box-shadow: 0 4px 15px rgba(99, 102, 241, 0.35) !important;
    transition: all 0.3s ease !important;
    cursor: pointer !important;
}
button.primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 25px rgba(99, 102, 241, 0.5) !important;
    filter: brightness(1.08);
}
button.primary:active {
    transform: translateY(0) !important;
}

/* ── Secondary / Reset Button ────────────────────────────────────────────── */
button.secondary {
    background: rgba(30, 32, 58, 0.7) !important;
    border: 1px solid rgba(139, 135, 230, 0.25) !important;
    border-radius: 12px !important;
    color: #c4b5fd !important;
    font-family: 'Inter', sans-serif !important;
    font-weight: 500 !important;
    font-size: 0.9rem !important;
    padding: 12px 24px !important;
    transition: all 0.3s ease !important;
    cursor: pointer !important;
}
button.secondary:hover {
    background: rgba(99, 102, 241, 0.12) !important;
    border-color: rgba(139, 135, 230, 0.5) !important;
    color: #e2e8f0 !important;
}

/* ── Status Box ──────────────────────────────────────────────────────────── */
#status-box textarea {
    background: rgba(16, 185, 129, 0.06) !important;
    border: 1px solid rgba(16, 185, 129, 0.2) !important;
    border-radius: 10px !important;
    color: #6ee7b7 !important;
    font-family: 'JetBrains Mono', 'Fira Code', monospace !important;
    font-size: 0.82rem !important;
}

/* ── Chatbot ─────────────────────────────────────────────────────────────── */
.chatbot-container {
    border-radius: 16px !important;
    overflow: hidden;
}
#chatbot {
    background: rgba(10, 10, 30, 0.5) !important;
    border: 1px solid rgba(139, 135, 230, 0.12) !important;
    border-radius: 16px !important;
}
#chatbot .message {
    border-radius: 14px !important;
    padding: 14px 18px !important;
    font-size: 0.9rem !important;
    line-height: 1.6 !important;
    animation: fadeInMsg 0.3s ease-out;
}
@keyframes fadeInMsg {
    from { opacity: 0; transform: translateY(6px); }
    to   { opacity: 1; transform: translateY(0); }
}
#chatbot .bot {
    background: rgba(99, 102, 241, 0.1) !important;
    border: 1px solid rgba(99, 102, 241, 0.15) !important;
    color: #e2e8f0 !important;
}
#chatbot .user {
    background: rgba(139, 92, 246, 0.15) !important;
    border: 1px solid rgba(139, 92, 246, 0.2) !important;
    color: #f1f5f9 !important;
}

/* ── Question Input ──────────────────────────────────────────────────────── */
#question-input textarea {
    min-height: 52px !important;
    font-size: 0.92rem !important;
}

/* ── Footer ──────────────────────────────────────────────────────────────── */
#footer-note {
    text-align: center;
    padding: 18px 0 8px;
}
#footer-note p {
    color: #475569 !important;
    font-size: 0.75rem !important;
    letter-spacing: 0.5px;
}

/* ── Responsive ──────────────────────────────────────────────────────────── */
@media (max-width: 768px) {
    #app-header h1 { font-size: 1.5rem !important; }
    .panel-card { padding: 16px !important; border-radius: 14px !important; }
}

/* ── Scrollbar (Webkit) ──────────────────────────────────────────────────── */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb {
    background: rgba(139, 135, 230, 0.25);
    border-radius: 3px;
}
::-webkit-scrollbar-thumb:hover { background: rgba(139, 135, 230, 0.45); }
"""

# ──────────────────────────────────────────────────────────────────────────────
# 🎭  Custom Theme
# ──────────────────────────────────────────────────────────────────────────────
theme = gr.themes.Base(
    primary_hue=gr.themes.colors.indigo,
    secondary_hue=gr.themes.colors.purple,
    neutral_hue=gr.themes.colors.slate,
    font=gr.themes.GoogleFont("Inter"),
    font_mono=gr.themes.GoogleFont("JetBrains Mono"),
).set(
    body_background_fill="transparent",
    body_text_color="#e2e8f0",
    block_background_fill="transparent",
    block_border_width="0px",
    input_background_fill="rgba(15,15,35,0.6)",
    input_border_color="rgba(139,135,230,0.2)",
    input_border_width="1px",
    shadow_drop="none",
    shadow_spread="0px",
)

# ──────────────────────────────────────────────────────────────────────────────
# 🏗️  Layout
# ──────────────────────────────────────────────────────────────────────────────
with gr.Blocks(theme=theme, css=custom_css, title="PDF Analysis System") as demo:

    # ── Header ───────────────────────────────────────────────────────────────
    gr.HTML("""
        <div id="app-header">
            <h1>📄 Multi-PDF Document Analysis</h1>
            <p>Upload your documents, configure chunking, and ask intelligent questions</p>
        </div>
    """)

    with gr.Row(equal_height=False):

        # ── Left Panel — Controls ────────────────────────────────────────────
        with gr.Column(scale=1, min_width=320):
            with gr.Group(elem_classes="panel-card"):
                gr.Markdown("📁  DOCUMENT UPLOAD", elem_classes="section-title")
                file_upload = gr.File(
                    label="Drop PDFs here or click to browse",
                    file_types=[".pdf"],
                    file_count="multiple",
                    elem_classes="upload-zone",
                )

            with gr.Group(elem_classes="panel-card"):
                gr.Markdown("⚙️  PROCESSING SETTINGS", elem_classes="section-title")
                with gr.Row():
                    chunk_size_input = gr.Number(
                        label="Chunk Size",
                        value=800,
                        precision=0,
                        minimum=100,
                        maximum=4000,
                        info="Characters per text chunk",
                    )
                    chunk_overlap_input = gr.Number(
                        label="Overlap",
                        value=100,
                        precision=0,
                        minimum=0,
                        maximum=1000,
                        info="Overlap between chunks",
                    )
                process_btn = gr.Button(
                    "⚡  Process Documents",
                    variant="primary",
                    size="lg",
                )

            with gr.Group(elem_classes="panel-card"):
                gr.Markdown("📊  STATUS", elem_classes="section-title")
                status_box = gr.Textbox(
                    value="● Ready — upload documents to begin.",
                    interactive=False,
                    show_label=False,
                    lines=2,
                    elem_id="status-box",
                )

        # ── Right Panel — Chat ───────────────────────────────────────────────
        with gr.Column(scale=2, min_width=480):
            with gr.Group(elem_classes="panel-card chatbot-container"):
                gr.Markdown("💬  CONVERSATION", elem_classes="section-title")
                chatbot = gr.Chatbot(
                    label="Chat",
                    height=480,
                    show_label=False,
                    elem_id="chatbot",
                    avatar_images=(None, None)
                )
                question_box = gr.Textbox(
                    placeholder="Ask a question about your documents…",
                    show_label=False,
                    lines=1,
                    max_lines=4,
                    elem_id="question-input",
                    container=False,
                )
                with gr.Row():
                    ask_btn = gr.Button("🔍  Ask", variant="primary", scale=3)
                    clear_btn = gr.Button("🗑  Reset", variant="secondary", scale=1)

    # ── Footer ───────────────────────────────────────────────────────────────
    gr.HTML("""
        <div id="footer-note">
            <p>Built with Gradio &amp; LangChain — Multi-PDF RAG Pipeline</p>
        </div>
    """)

    # ──────────────────────────────────────────────────────────────────────────
    # 🔗  Event Bindings
    # ──────────────────────────────────────────────────────────────────────────
    process_btn.click(
        fn=process_pdfs,
        inputs=[file_upload, chunk_size_input, chunk_overlap_input],
        outputs=status_box,
    )

    ask_btn.click(
        fn=ask_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box],
    )

    question_box.submit(
        fn=ask_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box],
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[chatbot, question_box, file_upload, status_box],
    )

demo.launch(debug=True)

/tmp/ipykernel_1667/812565606.py:251: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="PDF Analysis System") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://92f19c211ff8e81c95.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_google_genai/embeddings.py", line 449, in embed_documents
    result = self.client.models.embed_content(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 6401, in embed_content
    return self._embed_content(model=model, contents=contents, config=config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5206, in _embed_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1650, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client